In [1]:
import os
import datetime
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import DataParallel
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import yaml
import mne
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import datetime as dt
from tqdm import tqdm
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor, Resize, Compose
from torchvision.utils import make_grid, save_image
from PIL import Image
import matplotlib.pyplot as plt

In [2]:
class Config:
    def __init__(self, z_dim, max_channels, max_size, preG_path, preD_path, data_dir, num_dataset, \
                 noise_mean, noise_std, device, lr, batch_size_list, epoches, current_step, current_epoch):
        self.net = type('', (), {})()
        self.net.z_dim = z_dim
        self.net.max_channels = max_channels
        self.net.max_size = max_size
        self.net.preG_path = preG_path
        self.net.preD_path = preD_path
        
        self.dataset = type('', (), {})()
        self.dataset.data_dir = data_dir
        self.dataset.num_dataset = num_dataset
        self.dataset.noise_mean = noise_mean
        self.dataset.noise_std = noise_std
        
        self.train = type('', (), {})()
        self.train.device = device
        self.train.lr = lr
        self.train.batch_size_list = batch_size_list
        self.train.epoches = epoches
        self.train.current_step = current_step
        self.train.current_epoch = current_epoch

In [3]:
mnist_cfg = Config(
    z_dim=448,
    max_channels=448,
    max_size=32,  # dimensione dell'immagine da generare
    # FFHQ checkpoints are not used and are not included in this repository.
    preG_path=None,
    preD_path=None,
    data_dir='../datasetMNIST',  # specifica la directory dei dati MNIST
    num_dataset=10000,  # numero di immagini usate dal dataset
    noise_mean=0,
    noise_std=1,
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'),
    lr=0.001,
    batch_size_list=[1024, 512, 128, 64],
    epoches=100,
    current_step=0,
    current_epoch=0
)

In [ ]:
import torch

# Verifica se CUDA è disponibile
if torch.cuda.is_available():
    print("CUDA è disponibile. Numero di GPU:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA non è disponibile.")

In [5]:
def get_default_args():
    args = {
        'data_dir': '../../eeg_raw_dataset/Digit',
        'batch_size': 64,
        'vocab_size': 32
    }
    return args

### Load EEG Data CHAR

In [6]:
def class2index(args):
    basename = os.path.basename(os.path.normpath(args['data_dir']))
    if basename ==  'Digit':
        cls_names = ['0','1','2','3','4','5','6','7','8','9']
    elif basename == 'Char':
        cls_names = ['a','c','f','h','j','m','p','s','t','y']
    elif basename == 'Image':
        cls_names = ['apple','car','dog','gold','mobile','rose','scooter','tiger','wallet','watch']
    else:
        raise Exception("Sorry, unrecognized dir name:" + basename) 
    class_to_index_dic = dict(zip(cls_names, range(len(cls_names))))
    print(class_to_index_dic)
    return class_to_index_dic

def file_name_to_cls_index(file_name, class_to_index_dic):
    name, _ = file_name.split('.')
    _,cls = name.split('_')
    return class_to_index_dic[cls.lower()]  

In [7]:
def read_kumar_raw(args, unsupervised=False):
    print("Dataset dir:", args['data_dir'])
    files = os.listdir(args['data_dir'])
    num_raw_files=len(files)

    # mapping class name to indices
    charcls2index = class2index(args)

    # some statistics on all the raw files
    ts_len = []
    ts_max = []
    for fi in files:
        data = mne.io.read_raw_edf(os.path.join(args['data_dir'],fi), verbose='CRITICAL')
        # read channels 'AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4', 'RAW_CQ'
        raw_data = data[2:16][0]
        n_channels, len_time_series = raw_data.shape
        ts_len.append(len_time_series)
        ts_max.append(np.max(raw_data))
    print(f"Lenght Time series: MIN={np.min(ts_len)}, MAX={np.max(ts_len)}")
    print(f"Max value in Time series: {np.max(ts_max)}")
    print(f"Num raw files: {num_raw_files}")
    print(f"Num channels: {n_channels}")

    # Read data and scale them
    MAX_LEN_TS = 1280
    SCALE = 1000
    assert MAX_LEN_TS <= np.min(ts_len), f"{MAX_LEN_TS} is not <= {np.min(ts_len)}"

    X = np.zeros((num_raw_files,n_channels,MAX_LEN_TS))
    Y = np.zeros((num_raw_files,))
    for idx, fi in enumerate(files):
        data = mne.io.read_raw_edf(os.path.join(args['data_dir'], fi), verbose='CRITICAL')
        raw_data_scaled = data[2:16][0]*SCALE
        X[idx,:,:] = raw_data_scaled[:,0:MAX_LEN_TS]
        Y[idx] = file_name_to_cls_index(fi, charcls2index)

    # sliding window on the source timeseries
    npt = args['vocab_size']
    X_new = np.zeros((36110, npt, n_channels))
    Y_new = np.zeros((36110,))

    stride = 8
    ctr = 0
    for i in range(0,num_raw_files):
        y = Y[i]
        a= X[i,:,:]
        a = a.transpose()
        val = 0
        while val<=(len(a)-npt):
            x = a[val:val+npt,:]
            X_new[ctr,:,:] = x
            Y_new[ctr] = y
            val = val+stride
            ctr = ctr+1

    mean, std = X_new.mean(), X_new.std()
    print("Input data: mean =", mean, ", std =", std) # used for input data Normalization
    print("Kumar Dataset shape:", X_new.shape, ", min:", np.min(X_new), ", max:", np.max(X_new))

    # Random split. It is better to split the 230 files before the sliding window...
    X_train, X_test, Y_train, Y_test = train_test_split(np.float32(X_new), Y_new.astype(int), test_size=0.2, random_state=1)   

    #RESHAPE IN [BATCH,32x14,1,1]
    X_train = X_train.reshape(-1, 32 * 14, 1, 1)
    X_test = X_test.reshape(-1, 32 * 14, 1, 1)

    # Convert NumPy arrays to PyTorch tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    Y_train = torch.tensor(Y_train, dtype=torch.long)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    Y_test = torch.tensor(Y_test, dtype=torch.long)
    print("Training shape:", X_train.shape)
    print("Test shape:", X_test.shape)

    # Convert data to DataLoader
    train_dataset = TensorDataset(X_train, Y_train)
    train_loader = DataLoader(train_dataset, batch_size=args['batch_size'], shuffle=True)
    test_dataset = TensorDataset(X_test, Y_test)
    test_loader = DataLoader(test_dataset, batch_size=args['batch_size'], shuffle=False)
    return train_loader, test_loader


In [ ]:
args = get_default_args()
train_loader, test_loader = read_kumar_raw(args) 
print("Training size:", len(train_loader.dataset))
print("Test size:", len(test_loader.dataset))
# Itera sui dati del trainloader
for batch_idx, (data, target) in enumerate(train_loader):
    # Supponiamo che 'data' contenga i valori EEG con dimensioni [batch_size, 448, 1, 1]
    # Assicurati che i valori siano in formato corretto
    eeg_values = data  # Se 'data' contiene i valori EEG direttamente
    
    # Controlla la forma dei valori EEG
    print(eeg_values.shape)  # Dovrebbe essere [batch_size, 448, 1, 1]
    
    # Usa i valori EEG come input z nel tuo modello
    z = eeg_values  # Questo sostituisce z con i valori EEG
    break

In [9]:
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor, Resize
from torch.utils.data import Dataset, Subset

class MNISTDataset(Dataset):
    def __init__(self, data_dir, num=None, transforms=None):
        self.dataset = MNIST(root=data_dir, train=True, download=True, transform=transforms)
        self.num = num
        if num is not None and num < len(self.dataset):
            indices = np.random.choice(len(self.dataset), num, replace=False)
            self.dataset = Subset(self.dataset, indices)
       

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return img, label

In [10]:
dataset_mnist = MNISTDataset(mnist_cfg.dataset.data_dir)

In [11]:
img, label = dataset_mnist.__getitem__(0)

In [ ]:
label

In [ ]:
img

#### WSConv2d, PixelNorm, ConvBlock, ToRGB: non viene cambiato nulla

In [14]:
class WSConv2d(nn.Module):
    """
    Weight scaled Conv2d (Equalized Learning Rate)
    Note that input is multiplied rather than changing weights
    this will have the same result.
    Inspired and looked at:
    https://github.com/nvnbny/progressive_growing_of_gans/blob/master/modelUtils.py
    """

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, gain=2):
        super(WSConv2d, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.scale = (gain / (in_channels * (kernel_size ** 2))) ** 0.5 # **0,5 indica che tutto è sotto radice quadrata
        self.bias = self.conv.bias
        self.conv.bias = None

        # initialize conv layer
        nn.init.normal_(self.conv.weight)
        nn.init.zeros_(self.bias)

    def forward(self, x):
        return self.conv(x * self.scale) + self.bias.view(1, self.bias.shape[0], 1, 1)
    
class PixelNorm(nn.Module):
    def __init__(self, eps=1e-8):
        super(PixelNorm, self).__init__()
        self.eps = eps
        
    def forward(self, x):
        return x / torch.sqrt(torch.mean(x ** 2, dim=1, keepdim=True) + self.eps)
    
# (Conv3*3 + LReLU + PixelNorm) * 2
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.block = nn.Sequential(
            WSConv2d(in_channels, out_channels, 3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            PixelNorm(),
            WSConv2d(out_channels, out_channels, 3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            PixelNorm(),
        )
    
    def forward(self, x):
        return self.block(x)
    
#usato nel Generator
class ToRGB(nn.Module):
    def __init__(self, in_channels):
        super(ToRGB, self).__init__()
        self.rgb = WSConv2d(in_channels, 3, 1)
        
    def forward(self, x):
        return self.rgb(x)

#### Generatore: cambio valore z_dim da 512 a 448 dato che segnali eeg sono in dimensione 32x14 = 448

In [15]:
#input del Generator: dati eeg
#output: immagine sintetica (ricreata)
class Generator(nn.Module):
    def __init__(self, z_dim=448, in_channels=448, max_size=128):
        super(Generator, self).__init__()
        self.input = nn.Sequential(
            PixelNorm(),
            nn.ConvTranspose2d(z_dim, in_channels, 4), # da block 1x1 -> 4x4
            nn.LeakyReLU(0.2, inplace=True),
            PixelNorm(),
            WSConv2d(in_channels, in_channels, 3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            PixelNorm(),
        ) # out: in_channels * 4 * 4
        
        steps = int(np.log2(max_size) - 2)
        self.blocks = nn.ModuleList([])
        self.rgbs = nn.ModuleList([ToRGB(in_channels)])
        channels = in_channels
        for i in range(steps):
            if i < 3:
                self.blocks.append(ConvBlock(channels, channels))
                self.rgbs.append(ToRGB(channels))
            else:
                self.blocks.append(ConvBlock(channels, channels//2))
                self.rgbs.append(ToRGB(channels//2))
                channels = channels // 2
                
    def fade_in(self, block, up, alpha):
        return torch.tanh(alpha * block + (1 - alpha) * up)
    
    def forward(self, z, step, alpha):
        x = self.input(z)
        if step == 0:
            return self.rgbs[step](x)
        
        for i in range(step-1):
            x = F.interpolate(x, scale_factor=2, mode='nearest')
            x = self.blocks[i](x)
        
        x1 = F.interpolate(x, scale_factor=2, mode='nearest')
        block = self.rgbs[step](self.blocks[step-1](x1))
        
        up = self.rgbs[step-1](x)
        up = F.interpolate(up, scale_factor=2, mode='nearest')
        
        return self.fade_in(block, up, alpha)

#### minibatch_std, FromRGB, Discriminatore, load_dict: non cambia nulla

In [16]:
#calcola deviazione standard delle feature map del minibatch x e concatena
#le deviazioni standard calcolate per ogni feature map.
def minibatch_standard_deviation(x):
    mean_std = torch.std(x, dim=0).mean()
    mean_std = mean_std.repeat(x.shape[0], 1, x.shape[2], x.shape[3])
    return torch.cat([x, mean_std], dim=1)
    
#usato nel Discriminator        
class FromRGB(nn.Module):
    def __init__(self, out_channels):
        super(FromRGB, self).__init__()
        self.rgb = WSConv2d(3, out_channels, 1)
        
    def forward(self, x):
        return self.rgb(x)

#input: una immagine
#output: stima della sua autenticità
class Discriminator(nn.Module):
    def __init__(self, max_size=128, out_channels=448):
        super(Discriminator, self).__init__()
        
        self.out = nn.Sequential(
            WSConv2d(out_channels+1, out_channels, 3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            WSConv2d(out_channels, out_channels, 4)
        )
        self.linear = nn.Linear(out_channels, 1)
        
        num_blocks = int(np.log2(max_size) - 2)
        self.blocks = nn.ModuleList([])
        self.from_rgbs = nn.ModuleList([FromRGB(out_channels)])
        channels = out_channels
        for i in range(num_blocks):
            if i < 3:
                self.from_rgbs.append(FromRGB(channels))
                self.blocks.append(ConvBlock(channels, channels))
            else:
                self.from_rgbs.append(FromRGB(channels//2))
                self.blocks.append(ConvBlock(channels//2, channels))
                channels = channels // 2
        
        self.down = nn.AvgPool2d(kernel_size=2, stride=2)
        
    def fade_in(self, block, down, alpha):
        return torch.tanh(alpha * block + (1 - alpha) * down)
    
    def forward(self, x, step, alpha):
        if step == 0:
            x = self.from_rgbs[0](x)
            x = minibatch_standard_deviation(x)
            x = self.out(x).view(x.shape[0], -1)
            return self.linear(x)
        
        block = self.down(self.blocks[step-1](self.from_rgbs[step](x)))
        down = self.from_rgbs[step-1](self.down(x))
        x = self.fade_in(block, down, alpha)
        
        for i in range(step-2, -1, -1):
            x = self.down(self.blocks[i](x))
            
        x = minibatch_standard_deviation(x)
        x = self.out(x).view(x.shape[0], -1)
        return self.linear(x)

#funzione che carica un dizionario di pesi pre-addestrati.
#Utilizzata per inizializzare pesi del Generatore e Discriminatore con pesi pre-addestrati
def load_dict(net, model_dict, device=None):
    net_dict = net.state_dict()
    state_dict = {k:v for k,v in model_dict.items() if k in net_dict.keys()}
    net_dict.update(state_dict)
    net.load_state_dict(net_dict)
    if device is not None:
        net = net.to(device)
    return net

#### ProWGANDLoss, TriplicateChannel, ColorJitterChannels (non usata): non cambia nulla

In [17]:
#Implementazione personalizzata della loss function usata nel training con Gradient penalty
class ProWGANDLoss(nn.Module):
    def __init__(self):
        super(ProWGANDLoss, self).__init__()
    """
    calcola il gradient penalty per migliorare stabilità della rete neurale
    Viene utilizzato per evitare che il gradient del Discriminator non diventi
    troppo grande evitando il collasso del training.
    Calcola la norma dei gradient delle feature map di input rispetto la loro media, penalizzando
    il discriminatore se la norma si discosta dalla norma desiderata.
    """
    def gradient_penalty(self, D, real_img, fake_img, step, alpha, LAMDA=10):
        batch_size = real_img.size(0)
        device = real_img.device
        gp_alpha = torch.rand(batch_size, 1)
        gp_alpha = gp_alpha.expand(batch_size, real_img.nelement()//batch_size).reshape(real_img.shape).to(device)
        x = (gp_alpha * real_img + (1 - gp_alpha) * fake_img).requires_grad_(True).to(device)
        out = D(x, step=step, alpha=alpha)

        grad_outputs = torch.ones(out.shape).to(device)

        gradients = torch.autograd.grad(outputs=out, inputs=x, grad_outputs=grad_outputs, create_graph=True, only_inputs=True)[0]
        gradients = gradients.reshape(batch_size, -1)

        return LAMDA * ((gradients.norm(2, dim=1)-1)**2).mean()
    
    def forward(self, G, D, z, img, step, alpha):
        # D loss sia sui dati reali sia sui dati falsi generati dal Generator.
        # aggiunge anche la gradient penalty al Discriminatore.
        loss_real = -D(img, step=step, alpha=alpha).mean()
        fake_img = G(z, step=step, alpha=alpha)
        loss_fake = D(fake_img.detach(), step=step, alpha=alpha).mean()
        gp = self.gradient_penalty(D, img.detach(), fake_img.detach(), step=step, alpha=alpha, LAMDA=10)
        
        #la loss totale è la somma della loss sulle immagini reali, su quelle fake e la gradient penalty (gp)
        return loss_real + loss_fake + gp
    
class TriplicateChannels(object):
    def __call__(self, img):
        return img.repeat(3, 1, 1)
    
class ColorJitterChannels(object):
    def __init__(self, jitter_amount=0.2):
        self.jitter_amount = jitter_amount

    def __call__(self, img):
        # Supponiamo che img sia un tensor con forma [3, H, W]
        channels = []
        for c in range(3):
            jitter = torch.empty(1).uniform_(-self.jitter_amount, self.jitter_amount).item()
            channels.append(img[c] + jitter)
        return torch.stack(channels)

### Class Trainer: 
- non si usano i modelli preaddestrati. si addestra da 0
- vettore rumore casuale z viene sostituito dai segnali eeg
- le immagini usate dal discriminatore sono quelle del dataset Thoughtviz, in questo caso SOLO i caratteri (char)

In [18]:
from torchvision import transforms
        
class Trainer:
    def __init__(self, cfg, dataset):
        print(f'Load config')
        self.cfg = cfg
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f'Load dataset')
        self.dataset = dataset
        
        print('Init G and D models')
        self.G = Generator(z_dim=self.cfg.net.z_dim, in_channels=self.cfg.net.max_channels, max_size=self.cfg.net.max_size).to(self.cfg.train.device)
        self.D = Discriminator(max_size=self.cfg.net.max_size, out_channels=self.cfg.net.max_channels).to(self.cfg.train.device)
        
        # Using DataParallel to use multiple GPUs
        if torch.cuda.device_count() > 1:
            print(f'Using {torch.cuda.device_count()} GPUs')
            self.G = DataParallel(self.G, device_ids=[0,2])
            self.D = DataParallel(self.D, device_ids=[0,2])
            #self.G = DataParallel(self.G)
            #self.D = DataParallel(self.D)
        
        # pretrained
        if self.cfg.net.preG_path is not None:
            print('Load pretrained G model')
            self.G = load_dict(self.G, torch.load(self.cfg.net.preG_path), device=self.cfg.train.device)
        if self.cfg.net.preD_path is not None:
            print('Load pretrained D model')
            self.D = load_dict(self.D, torch.load(self.cfg.net.preD_path), device=self.cfg.train.device)
        
        print('Set optimizers and loss function')
        self.optimizer_G = torch.optim.Adam(self.G.parameters(), lr=self.cfg.train.lr, betas=(0.0, 0.99))
        self.optimizer_D = torch.optim.Adam(self.D.parameters(), lr=self.cfg.train.lr, betas=(0.0, 0.99))
        self.lossfn_D = ProWGANDLoss().to(self.cfg.train.device)
        
        self.fixed_z = torch.normal(self.cfg.dataset.noise_mean, self.cfg.dataset.noise_std, size=(64, self.cfg.net.z_dim, 1, 1)).to(self.cfg.train.device)
        
        # output
        self.imgs = []
        self.output_dir = './images'  # Directory di output predefinita
        os.makedirs(self.output_dir, exist_ok=True)  # Creazione della directory se non esiste
        print('Initial done')
        
        # File di log contenente le loss ad ogni epoca
        self.log_file = os.path.join('./logs', 'training_log.txt')
        os.makedirs(os.path.dirname(self.log_file), exist_ok=True)
        
    def get_loader(self, step):
        img_size = 2 ** (step + 2)
        print(f"Image size: {img_size}")
        
        transforms_ = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size, img_size), antialias=True),
            TriplicateChannels(), #Trasformazione per triplicare i canali
            #ColorJitterChannels(jitter_amount=0.2)  # Aggiungi questa trasformazione per il jitter di colore
        ])
        
        img_dataset = self.dataset(data_dir=self.cfg.dataset.data_dir, transforms=transforms_, num=self.cfg.dataset.num_dataset)
        print(f"Dataset length: {len(img_dataset)}")
        print(f"Batch size: {self.cfg.train.batch_size_list[step]}")  # Stampa la dimensione del batch
        data_loader = DataLoader(
            img_dataset,
            batch_size=self.cfg.train.batch_size_list[step],
            shuffle=False,
            num_workers=2
        )
        print(f"DataLoader length: {len(data_loader)}")  # Stampa la lunghezza del DataLoader
        
        # Itera su un batch per ottenere il tensor e stampare la sua dimensione e forma
        for img, label in data_loader:
            print(f"Tensor shape: {img.shape}")  # Stampa la forma del tensor
            print(f"Tensor size: {img.size()}")  # Stampa la dimensione del tensor
            print(f"Label: {label}")  # Stampa la label
            # Calcolo dei valori minimo, massimo e media
            #min_val = torch.min(data).item()
            #max_val = torch.max(data).item()
            #mean_val = torch.mean(data).item()
            #print(f'Min: {min_val}, Max: {max_val}, Mean: {mean_val}')
            break  # Rompiamo il loop dopo il primo batch per evitare stampe ripetute
        
        
        return len(img_dataset), data_loader
        
    ####++++++++++++ Metodo per generare e stampare le immagini ad ogni epoca A GRIGLIA +++++++++++++#####
    """def generate_and_print_images(self, step, alpha):
        with torch.no_grad():
            generated_images = self.G(self.fixed_z, step=step, alpha=alpha).cpu()
            print(f"Generated images shape: {generated_images.shape}")  # Stampa la dimensione delle immagini generate
            out_img = make_grid(generated_images, padding=2, normalize=True).permute(1,2,0).numpy()
        out_img = (out_img - np.min(out_img)) / (np.max(out_img) - np.min(out_img)).astype(float)
        plt.imshow(out_img)
        plt.axis('off')
        plt.show()"""
        
    ######+++++++++++ Metodo per generare e stampare le immagini ad ogni epoca SINGOLARMENTE ++++++++#####
    def generate_and_print_images(self, step, alpha, labels):
        self.G.eval()
        with torch.no_grad():
            generated_images = self.G(self.fixed_z, step=step, alpha=alpha).cpu()
            print(f"Generated images shape: {generated_images.shape}")  # Stampa la dimensione delle immagini generate
            #seleziono le prime 10 di 64 immagini di output
            selected_images = generated_images[:10]
            selected_labels = labels[:10]
            #imposto il subplots per mostrare le immagini in un unico output e non 10 uno sotto l'altro
            fig, axes = plt.subplots(1, 10, figsize=(10 * 3, 3))
            for i, (image,ax, label) in enumerate(zip(selected_images, axes, selected_labels)):
                out_img = image.permute(1, 2, 0).numpy()
                out_img = (out_img - np.min(out_img)) / (np.max(out_img) - np.min(out_img)).astype(float)

                ax.imshow(out_img)
                ax.axis('off')
                ax.set_title(f'Label: {label.item()}')
            plt.tight_layout()
            plt.show()
        self.G.train()
    
    def train(self):
        total_steps = int(np.log2(self.cfg.net.max_size) - 1)
        assert self.cfg.train.current_step < total_steps, f'current_step of config must less than {total_steps}'
        print(f'Start training at steps: {self.cfg.train.current_step+1}/{total_steps}')
        
        for step in range(self.cfg.train.current_step, total_steps):
            num_dataset, loader = self.get_loader(step)
            start_epoch = self.cfg.train.current_epoch if self.cfg.train.current_step == step else 0
            alpha = start_epoch * 2 / self.cfg.train.epoches if self.cfg.train.current_step == step else 1e-8
            for epoch in range(start_epoch, self.cfg.train.epoches):
                begin_time = time.time()
                for i, (img, label) in enumerate(tqdm(loader, desc=f'epoch {epoch+1}/{self.cfg.train.epoches}')):
                    z = torch.normal(self.cfg.dataset.noise_mean, self.cfg.dataset.noise_std, size=(img.size(0), self.cfg.net.z_dim, 1, 1))
                    img, z = img.to(self.cfg.train.device), z.to(self.cfg.train.device)
                    #print(z.shape)
                    #print(img.shape)
                    # train D
                    self.D.zero_grad()
                    loss_D = self.lossfn_D(self.G, self.D, z, img, step, alpha)
                    loss_D.backward()
                    self.optimizer_D.step()

                    # train G
                    self.G.zero_grad()
                    loss_G = -self.D(self.G(z, step=step, alpha=alpha), step=step, alpha=alpha).mean()
                    loss_G.backward()
                    self.optimizer_G.step()
                    
                    # smooth increase alpha
                    # it reaches 1 after half of epoches
                    alpha += 2 * img.size(0) / (self.cfg.train.epoches * num_dataset)
                    alpha = min(alpha, 1)
                    
                end_time = time.time()
                self.log(begin_time, end_time, step, total_steps, epoch, alpha, loss_G, loss_D)
                # Salva l'immagine generata
                with torch.no_grad():
                    out_img = make_grid(self.G(self.fixed_z, step=step, alpha=alpha), padding=2, normalize=True)
                    save_image(out_img, os.path.join(self.output_dir, f'step_{step}_epoch_{epoch}.png'))
                
                self.generate_and_print_images(step, alpha, label)  # Genera e stampa le immagini ad ogni epoca
        print('Train done')
    
    def log(self, begin_time, end_time, step, total_steps, epoch, alpha, loss_G, loss_D):
        out_str = '[total time: {:.5f}s] '.format(end_time-begin_time) + f'[Step: {step+1}/{total_steps}] [Epoch: {epoch+1}/{self.cfg.train.epoches}] [alpha: {format(alpha, ".2e")}] [G loss: {loss_G.item():.4f}] [D loss: {loss_D.item():.4f}]'
        print(out_str)
        # Salva i dati su file
        with open(self.log_file, 'a') as f:
            f.write(out_str + '\n')

#### configurazione di parametri della rete neurale:


In [19]:
mnist_trainer = Trainer(cfg=mnist_cfg, dataset=MNISTDataset)
mnist_trainer.train()